In [ ]:
# 1. Verificar GPU
!nvidia-smi

Mon Nov 17 20:04:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# install_deps.ipynb
!pip install mediapipe opencv-python-headless numpy torch torchvision scikit-learn tqdm

In [ ]:
# clone_dataset.ipynb
!git clone https://github.com/bleon133/DataSet_LSC.git
%cd DataSet_LSC
!ls -l

Cloning into 'DataSet_LSC'...
remote: Enumerating objects: 565, done.
remote: Total 565 (delta 0), reused 0 (delta 0), pack-reused 565 (from 1)
Receiving objects: 100% (565/565), 1.02 GiB | 49.78 MiB/s, done.
Updating files: 100% (526/526), done.
/content/DataSet_LSC
total 164
drwxr-xr-x 2 root root 4096 Nov 17 20:04 adulto
drwxr-xr-x 2 root root 4096 Nov 17 20:04 algunos
drwxr-xr-x 2 root root 4096 Nov 17 20:04 anciano
drwxr-xr-x 2 root root 4096 Nov 17 20:04 año
drwxr-xr-x 2 root root 4096 Nov 17 20:04 balon
drwxr-xr-x 2 root root 4096 Nov 17 20:04 bebe
drwxr-xr-x 2 root root 4096 Nov 17 20:04 buenas-noches
drwxr-xr-x 2 root root 4096 Nov 17 20:04 buenos-dias
drwxr-xr-x 2 root root 4096 Nov 17 20:04 bus
drwxr-xr-x 2 root root 4096 Nov 17 20:04 cada-uno
drwxr-xr-x 2 root root 4096 Nov 17 20:04 casas
drwxr-xr-x 2 root root 4096 Nov 17 20:04 comprar
drwxr-xr-x 2 root root 4096 Nov 17 20:04 dormir
drwxr-xr-x 2 root root 4096 Nov 17 20:04 el-ella
drwxr-xr-x 2 root root 4096 Nov 17 20:04 e

In [ ]:
# preprocess_all_videos.ipynb
import os
import cv2
import mediapipe as mp
import numpy as np
import pickle
from tqdm import tqdm
import json
from sklearn.preprocessing import LabelEncoder

# ================= CONFIGURACIÓN =================
DATASET_PATH = "/content/DataSet_LSC"
OUTPUT_PATH = "/content/processed_data"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Número de frames por video (fijo)
NUM_FRAMES = 30

# Lista de glosas (tus 40 palabras)
GLOSAS = [
    "adulto", "algunos", "anciano", "año", "balon", "bebe", "buenas-noches", "buenos-dias",
    "bus", "cada-uno", "casas", "comprar", "dormir", "el-ella", "ellas-ellos", "ellos-dos-ellas-dos",
    "futbol", "futuro", "hambre", "hijo", "hola", "hombre", "ir", "jugar", "llegar", "mujer",
    "nacer", "necesitar", "otro", "pais", "pasado", "pedalear-bicleta", "persona", "reunion",
    "sordo", "tarde", "un", "ver", "visitar", "ya", "yo"
]

print(f"Procesando {len(GLOSAS)} glosas...")

# ================= MEDIAPIPE =================
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

pose = mp_pose.Pose(
    static_image_mode=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ================= FUNCIÓN DE EXTRACCIÓN =================
def extract_keypoints_from_video(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return None

    # Seleccionar índices uniformes
    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    keypoints_sequence = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        # Convertir a RGB
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        hand_results = hands.process(image_rgb)
        pose_results = pose.process(image_rgb)

        # === POSE: 33 landmarks x 3 coords = 99 valores ===
        pose_vector = np.zeros(33 * 3)
        if pose_results.pose_landmarks:
            for i, lm in enumerate(pose_results.pose_landmarks.landmark):
                pose_vector[i*3:(i*3)+3] = [lm.x, lm.y, lm.z]

        # === MANOS: 2 manos x 21 landmarks x 3 = 126 valores ===
        hand_vector = np.zeros(2 * 21 * 3)
        if hand_results.multi_hand_landmarks:
            for hand_idx, hand_landmarks in enumerate(hand_results.multi_hand_landmarks[:2]):
                for i, lm in enumerate(hand_landmarks.landmark):
                    offset = hand_idx * 21 * 3 + i * 3
                    hand_vector[offset:offset+3] = [lm.x, lm.y, lm.z]

        # === CONCATENAR: pose + manos = 99 + 126 = 225 features ===
        kp_vector = np.concatenate([pose_vector, hand_vector])
        keypoints_sequence.append(kp_vector)

    cap.release()
    return np.array(keypoints_sequence) if len(keypoints_sequence) == num_frames else None

# ================= PROCESAMIENTO MASIVO =================
all_sequences = []
all_labels = []

print("Iniciando extracción de keypoints...\n")

for glosa in GLOSAS:
    folder_path = os.path.join(DATASET_PATH, glosa)
    if not os.path.exists(folder_path):
        print(f"⚠️ No existe carpeta: {glosa}")
        continue

    video_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.mp4', '.avi', '.mov'))]
    if len(video_files) == 0:
        print(f"⚠️ Sin videos en: {glosa}")
        continue

    print(f"Procesando '{glosa}' → {len(video_files)} videos")

    for video_file in tqdm(video_files, desc=f"{glosa}", leave=False):
        video_path = os.path.join(folder_path, video_file)
        seq = extract_keypoints_from_video(video_path)
        if seq is not None:
            all_sequences.append(seq)
            all_labels.append(glosa)

# ================= GUARDAR DATOS =================
X = np.array(all_sequences)
y = np.array(all_labels)

print(f"\nDataset final:")
print(f"   Secuencias: {X.shape} → (videos, 30 frames, 225 keypoints)")
print(f"   Etiquetas: {y.shape}")
print(f"   Clases únicas: {len(np.unique(y))}")

# Codificar etiquetas
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Guardar datos
data_to_save = {
    "X": X,
    "y": y_encoded,
    "label_names": le.classes_.tolist(),
    "glosas": GLOSAS
}

with open(os.path.join(OUTPUT_PATH, "lsc_dataset.pkl"), "wb") as f:
    pickle.dump(data_to_save, f)

# Guardar mapeo de etiquetas
with open(os.path.join(OUTPUT_PATH, "label_map.json"), "w") as f:
    json.dump({int(i): label for i, label in enumerate(le.classes_)}, f, indent=2)

print(f"\nPREPROCESAMIENTO COMPLETADO")
print(f"Archivo guardado: {OUTPUT_PATH}/lsc_dataset.pkl")
print(f"Mapeo: {OUTPUT_PATH}/label_map.json")

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


Procesando 41 glosas...
Iniciando extracción de keypoints...

Procesando 'adulto' → 20 videos


Procesando 'algunos' → 17 videos


Procesando 'anciano' → 16 videos


Procesando 'año' → 17 videos


...

Procesando 'ver' → 14 videos


Procesando 'visitar' → 14 videos


Procesando 'ya' → 13 videos


Procesando 'yo' → 14 videos



Dataset final:
   Secuencias: (490, 30, 225) → (videos, 30 frames, 225 keypoints)
   Etiquetas: (490,)
   Clases únicas: 41

PREPROCESAMIENTO COMPLETADO
Archivo guardado: /content/processed_data/lsc_dataset.pkl
Mapeo: /content/processed_data/label_map.json


In [ ]:
# download_files.ipynb
from google.colab import files

files.download("/content/processed_data/lsc_dataset.pkl")
files.download("/content/processed_data/label_map.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# verify_data.ipynb
import pickle
with open("/content/processed_data/lsc_dataset.pkl", "rb") as f:
    data = pickle.load(f)

print(data["X"].shape)  # Debe ser (N, 30, 225)
print(len(data["label_names"]))  # Debe ser 40

(490, 30, 225)
41
